# Sorting Data with SQL ORDER BY

## Introduction: What is Sorting?

In previous chapters, you learned how to retrieve specific columns using `SELECT`, filter rows using `WHERE`, and compute calculated fields using transformations and functions. 

Often, query output is returned in a random order. To analyze financial trends, rank operational performance, or prepare professional audit schedules, you must be able to arrange your records logically.

In data analytics, **sorting** means to rearrange items in a specified order based on the values in one or more columns.

Before writing a single line of SQL code, we must establish a fundamental principle of structured data: **Sorting rearranges entire rows.** A database table consists of structured records (rows) made up of individual fields (columns). When you sort a table by a specific field, that field acts as a **key**. The database engine moves every row as an intact unit. You are not simply sorting a single column of numbers or text; you are reordering the entire observation across all of its attributes. Thus, the data in every row is preserved during the sorting process. For example, if a row contains a customer ID and a customer name, sorting will preserve the ID associated with each customer.

### A Toy Dataset: Pets

To see how sorting works visually, consider a simple dataset describing sixteen pets on a farm. Each row represents a single pet (an observation) with four attributes: `Name`, `Type` (species), `Weight` (in kilograms), and `IsIndoorPet` (a boolean value indicating whether the pet lives indoors):

| Name | Type | Weight | IsIndoorPet |
| :--- | :--- | :--- | :--- |
| Duke | Dog | 35 | FALSE |
| Killer | Dog | 25 | FALSE |
| Lucy | Dog | 5 | TRUE |
| Oliver | Cat | 2 | TRUE |
| Milo | Cat | 10 | FALSE |
| Barbie | Horse | 500 | FALSE |
| Nero | Horse | 420 | FALSE |
| Donkey Hoté | Donkey | 225 | FALSE |
| Dave | Donkey | 210 | FALSE |
| Hazel | Donkey | 240 | FALSE |
| Azure | Parakeet | 0.036 | TRUE |
| Wilbur | Pig | 45 | FALSE |
| Stella | Pig | 32 | FALSE |
| Skunky | Skunk | 1 | FALSE |
| Bruce | Fish | 0.105 | TRUE |
| Crystal | Fish | 0.1 | TRUE |

Notice what happens if we sort this table alphabetically by `Name`:

```{figure} imgs/Animation_of_Pets_Selection_Sort.gif
:width: 85%
:align: center
:alt: Selection Sort Animation for Pets Dataset

Selection Sort Animation — Rearranging Rows by Name
```

As the sorting algorithm scans the dataset, it identifies the row containing `"Azure"` and moves that **entire record**—including her species (`Parakeet`), her weight (`0.036 kg`), and her indoor status (`TRUE`)—to the top position. Azure's weight is never separated from her name. Preserving row integrity is what distinguishes relational sorting from disconnected array manipulation.

## Single-Level Sorting: Ascending vs. Descending Order

In SQL, sorting is accomplished using the **`ORDER BY`** clause. The basic syntax of a query with an `ORDER BY` clause is:

```sql
SELECT Column1, Column2, ...
FROM TableName
ORDER BY ColumnToSort [ASC | DESC]
```

The direction of the sort is controlled by two keywords:
* **Ascending Order (`ASC`):** Arranges data from smallest to largest.
  * **Numeric Values:** `1, 2, 3, 100...`
  * **Text Strings:** Alphabetical order (`A` to `Z`).
  * **Dates:** Chronological order (oldest date to newest date).
  * *Note:* Ascending is the default sort direction in SQL. If you omit the `ASC` keyword, SQL automatically sorts in ascending order.
* **Descending Order (`DESC`):** Arranges data from largest to smallest.
  * **Numeric Values:** `100, 3, 2, 1...`
  * **Text Strings:** Reverse alphabetical order (`Z` to `A`).
  * **Dates:** Reverse chronological order (newest date to oldest date).

### Single-Level Sort Examples on the Pets Dataset

To retrieve all pets sorted alphabetically by name (ascending):

```sql
SELECT Name, Type, Weight, IsIndoorPet
FROM Pets
ORDER BY Name
```

*Output (First 5 rows):*
| Name | Type | Weight | IsIndoorPet |
| :--- | :--- | :--- | :--- |
| Azure | Parakeet | 0.036 | TRUE |
| Barbie | Horse | 500 | FALSE |
| Bruce | Fish | 0.105 | TRUE |
| Crystal | Fish | 0.1 | TRUE |
| Dave | Donkey | 210 | FALSE |

To sort the pets in reverse alphabetical order by name, add the `DESC` keyword:

```sql
SELECT Name, Type, Weight, IsIndoorPet
FROM Pets
ORDER BY Name DESC
```

*Output (First 5 rows):*
| Name | Type | Weight | IsIndoorPet |
| :--- | :--- | :--- | :--- |
| Wilbur | Pig | 45 | FALSE |
| Stella | Pig | 32 | FALSE |
| Skunky | Skunk | 1 | FALSE |
| Oliver | Cat | 2 | TRUE |
| Nero | Horse | 420 | FALSE |

## Database Sorting vs. Spreadsheet Sorting: An Audit Perspective

When students first learn SQL sorting, a common question arises: *"Can running an `ORDER BY` query permanently change the order of rows in my database table?"*

The answer is an emphatic **NO**.

In a relational database, **a `SELECT` statement with an `ORDER BY` clause never alters or modifies the underlying data stored on disk.** 

A database table is a permanent storage structure. When you execute a query with `ORDER BY`, the database management system reads the records, constructs a temporary result in memory, and displays that result on your screen. The raw database records remain completely untouched.

```{warning}
**The Auditor's Perspective: Spreadsheet Data Corruption vs. Database Integrity**

In Microsoft Excel, sorting physically moves cells on the worksheet grid and modifies the saved file on disk. Inexperienced accountants frequently commit a catastrophic data corruption error in Excel: they highlight a single column (e.g., Column C) and click "Sort" without expanding the selection to adjacent columns.

When this occurs in Excel, the spreadsheet sorts Column C in isolation while leaving Columns A, B, and D untouched! This completely severs the alignment between records—for instance, assigning Barbie the Horse's weight of 500 kg to Azure the Parakeet.

In professional accounting, this type of spreadsheet error leads to false financial statements, broken audit trails, and severe control failures. 

In SQL, this user error is impossible. Because the `ORDER BY` clause operates on the entire rows, SQL always moves entire rows together, guaranteeing 100% data integrity.
```

## Multi-Level Sorting

A **single-level sort** orders records using values from a single column. However, real-world datasets frequently contain repeated values within a column. 

For example, if you sort the `Pets` dataset by `Type` (species), you will get three dogs, two cats, three donkeys, two horses, two pigs, and two fish. How should SQL arrange the rows *within* the "Dog" category or *within* the "Cat" category?

To resolve ties across categorical groups, you perform a **multi-level sort**.

### Defining a Multi-Level Sort

A **multi-level sort** orders data by a primary column first. Whenever two or more rows contain identical values in the primary column, SQL uses a secondary column as a "tie-breaker" to determine their relative order. If ties remain in the secondary column, SQL can evaluate a tertiary column, and so on.

### Why Multi-Level Sorting Cannot Be Achieved Through Multiple Single-Level Sorts

A common misconception among budding data analysts is that a multi-level sort can be accomplished by executing sequential single-level sorts one after another (for example, sorting by `Weight` first, and then running a second sort by `Type`).

In computer programming, executing a new single-level sort **completely overwrites and destroys** the ordering established by any previous single-level sort! 

If you sort a dataset by `Weight` and then immediately apply a separate single-level sort by `Type`, the database re-sorts all rows strictly by `Type`. The prior sorting by `Weight` is discarded, leaving the items within each species in arbitrary disk order.

To see this, copy and paste the Pets dataset from above into Excel. Sort by `Type`. Then sort by `Weight`. Analyze the data and you will see that it is sorted by `Weight`. The sorting by type is lost. To perform a multi-level sort in Excel, you must use the *Sort* button and the dialog. See the following figure.

```{figure} imgs/ExcelMultiSort.png
:width: 65%
:align: center
:alt: Performing a multi-level sort in Excel

Performing a Multi-Level Sort in Excel
```

### Implementing Multi-Level Sorting in SQL

SQL handles multi-level sorting declaratively in a single, elegant clause. To perform a multi-level sort, pass a comma-separated list of column names to `ORDER BY`:

```sql
SELECT Name, Type, Weight, IsIndoorPet
FROM Pets
ORDER BY IsIndoorPet DESC, Type ASC, Weight DESC
```

Let's deconstruct how SQL evaluates this multi-level `ORDER BY` clause:
1. **Primary Sort (`IsIndoorPet DESC`):** SQL first divides all records into two main groups based on `IsIndoorPet`. Because we specified `DESC`, `TRUE` (indoor pets) appears first, followed by `FALSE` (outdoor pets).
2. **Secondary Sort (`Type ASC`):** Within the `TRUE` group (indoor pets), SQL resolves ties by sorting species alphabetically (`ASC`). It does the same within the `FALSE` group.
3. **Tertiary Sort (`Weight DESC`):** If two indoor pets belong to the same species (for example, Bruce the Fish and Crystal the Fish), SQL uses `Weight DESC` as a tie-breaker, placing the heavier fish (`Bruce`, `0.105 kg`) above the lighter fish (`Crystal`, `0.100 kg`).

### Multi-Level Sort Result

The table below shows the complete `Pets` dataset sorted by `IsIndoorPet DESC, Type ASC, Weight DESC`:

| Name | Type | Weight | IsIndoorPet |
| :--- | :--- | :--- | :--- |
| **Milo** | Cat | 10 | **TRUE** | 
| **Oliver** | Cat | 2 | **TRUE** |
| **Lucy** | Dog | 5 | **TRUE** |
| **Bruce** | Fish | 0.105 | **TRUE** | 
| **Crystal** | Fish | 0.1 | **TRUE** | 
| **Azure** | Parakeet | 0.036 | **TRUE** |
| **Duke** | Dog | 35 | **FALSE** | 
| **Killer** | Dog | 25 | **FALSE** | 
| **Hazel** | Donkey | 240 | **FALSE** | 
| **Donkey Hoté** | Donkey | 225 | **FALSE** | 
| **Dave** | Donkey | 210 | **FALSE** | 
| **Barbie** | Horse | 500 | **FALSE** | 
| **Nero** | Horse | 420 | **FALSE** | 
| **Wilbur** | Pig | 45 | **FALSE** | 
| **Stella** | Pig | 32 | **FALSE** | 
| **Skunky** | Skunk | 1 | **FALSE** | 

Notice how clean and structured the data becomes. Every category is grouped logically, and every tie within a category is sorted deterministically.

## Answering Business Questions Quickly by Sorting

In business analytics, sorting is not merely a formatting tool for final reports—it is one of the fastest ways to perform **exploratory data analysis**. By combining `ORDER BY` with `WHERE` and `TOP (n)`, auditors and financial analysts can answer critical business questions in seconds without writing complex formulas.

To see this in action, let's examine corporate 10-K financial data from **Compustat** (`CompanyData` table). This dataset contains **11,921 firm-year observations** spanning fiscal years 2016 through 2018 for U.S. public companies audited by Big 4 or major national accounting firms.

Key fields in `CompanyData` include:
* `GVKEY`: Unique Compustat company identifier.
* `FiscalYear`: Fiscal reporting year (`2016`, `2017`, `2018`).
* `CompanyName`: Official registered company name.
* `Ticker`: Stock exchange ticker symbol.
* `AuditorName`: Independent audit firm (e.g., `PwC`, `Ernst & Young`, `Deloitte`, `KPMG`, `BDO Seidman`, `Grant Thornton`).
* `CommonSharesOutstanding`: Total shares of stock issued.
* `ClosingPrice`: Closing stock price on the last day of the fiscal year.
* `MarketCap`: Total market capitalization, calculated as `CommonSharesOutstanding * ClosingPrice`.

Let's work through four real-world analytical scenarios using this database:

<font color="red"><b>Show the first few rows of the data. Reword the above accordingly.</b></font>